# Case NL: guarantee sensitivity frontier (`fig: frontier`)

This notebook produces the two-panel trade-off figure for the Dutch case study:

- **Left panel:** minimum revenue loss and weighted rule count against the *household income floor* swept over $[0.90, 1.00]$ (budget band fixed at $\pm 1.5\%$).
- **Right panel:** the same outcomes against the *budget band* swept over $[\pm 0.5\%, \pm 3\%]$ (income floor fixed at $5\%$).

The marginal-pressure cap is held fixed at $65\%$ in both panels. Each sweep point is solved **twice**, varying exactly one guarantee:

1. **`budget_only`** — a single-objective solve minimizing revenue loss. This gives the *true minimum* revenue loss under the guarantees (the blue series in the figure), which is monotone in the tightness of the guarantee.
2. **`sequential`** — the same two-stage lexicographic optimization used for the Dutch reform in `fig09_10_dutch_case_study.ipynb` (budget first, then complexity with a bounded budget slack). This gives the reform's weighted rule count (the orange series) and its realized spend, which may drift from the stage-1 minimum by up to the budget tolerance.

A third sweep over the marginal-pressure cap (at status-quo floor and band) is included for later analysis. Points where the solver reports infeasibility are recorded and shown as a shaded region in the figure. All sweep tables are saved incrementally during the loops, and the full rules-and-rates table of every solved model is written to `./systems/frontier_runs/` so reforms can be inspected afterwards.

In [ ]:
## Load libraries
import numpy as np
import os
import time
import pandas as pd
import matplotlib.pyplot as plt
import TaxSolver as tx
from TaxSolver.data_wrangling.data_loader import DataLoader
from TaxSolver.constraints.budget_constraint import BudgetConstraint
from TaxSolver.constraints.income_constraint import IncomeConstraint
from TaxSolver.constraints.marginal_pressure_constraint import MarginalPressureConstraint
from TaxSolver.objective import SequentialMixedObjective, BudgetObjective
from TaxSolver.backend.gurobi_backend import GurobiBackend

# Import the NLD rule book from the paper folder (local stand-alone copy)
from nld_rule_book import NLDRuleBook, setup_nl_optimization, get_mutually_exclusive_constraints

from copy import deepcopy

In [ ]:
def load_nl_data(data_path: str) -> dict:
    """Load NL data with the appropriate column mappings."""
    dl = DataLoader(
        path=data_path,
        income_before_tax="income_before_tax",
        income_after_tax="income_after_tax",
        weight="weight",
        id="id",
        hh_id="hh_id",
        mirror_id="mirror_id",
        input_vars=[
            "i_number_of_kids_0_5",
            "i_number_of_kids_6_11",
            "i_number_of_kids_12_15",
            "i_number_of_kids_16_17",
            "i_number_of_kids",
            "i_monthly_rent",
            "i_assets",
            "i_partner_income",
            "i_other_income",
            "i_woz",
            "i_mortgage_interest",
        ],
        group_vars=["fiscal_partner", "partner_type_of_income"],
    )
    return dl.households


def run_nl_optimization(
    households: dict,
    max_marginal_pressure: float = 0.65,
    max_budget_increase: float = None,  # Max additional spending (government loses revenue)
    min_budget_increase: float = None,  # Min spending change (negative = must collect more)
    income_constraint_value: float = 0.05,
    include_tags: list = None,
    exclude_tags: list = None,
    k_groups: list = None,
    time_limit: int = 15 * 60,
    budget_tolerance: float = None,
    objective_mode: str = "sequential",
):
    """Run the NL tax optimization for one sweep point.

    Differs from the `fig09_10_dutch_case_study.ipynb` version in two ways
    needed for sweeping: an infeasible model does not raise (the unsolved
    solver is returned; check ``.solved``), and ``budget_tolerance`` for the
    second (complexity) objective stage can be overridden -- by default it is
    1% of current tax revenue but capped at the budget band itself, so the
    complexity stage cannot leave the swept band.
    """
    if include_tags is None:
        include_tags = ["basic"]  # Core rules
    if exclude_tags is None:
        exclude_tags = ["verzilverbaar", "protect_other_koopkracht_groups"]
    if k_groups is None:
        k_groups = NLDRuleBook.default_k_groups()

    print("Configuration:")
    print(f"  - include_tags: {include_tags}")
    print(f"  - exclude_tags: {exclude_tags}")
    print(f"  - k_groups: {k_groups}")
    print(f"  - max_marginal_pressure: {max_marginal_pressure}")
    print(f"  - max_budget_increase: {max_budget_increase}")
    print(f"  - min_budget_increase: {min_budget_increase}")
    print(f"  - income_constraint: {income_constraint_value}")

    backend = GurobiBackend()
    backend.model.setParam('TimeLimit', time_limit)
    backend.model.setParam('NumericFocus', 2)

    solver_households = deepcopy(households)
    tax_solver = tx.TaxSolver(households=solver_households, backend=backend)

    rules = setup_nl_optimization(
        tax_solver=tax_solver,
        k_groups=k_groups,
        include_tags=include_tags,
        exclude_tags=exclude_tags,
        add_main_brackets=True,   # 13 main brackets with weight=1
        add_k_group_brackets=True,  # 30 k-group brackets with weight=4
        add_household_brackets=False,  # Avoids double-taxing couples
    )
    print(f"\nAdded {len(rules)} total rules:")

    current_tax_revenue = sum(
        hh.members[0]["weight"] * sum(
            p["income_before_tax"] - p["income_after_tax"] for p in hh.members
        )
        for hh in solver_households.values()
    )
    print(f"  - current_tax_revenue (weighted): {current_tax_revenue:,.2f}")

    # max_budget_increase: positive = government can collect LESS (lose revenue)
    # min_budget_increase: negative = government can collect MORE (gain revenue)
    if max_budget_increase is None:
        max_budget_increase = current_tax_revenue * 0.015  # Allow 1.5% revenue decrease
    if min_budget_increase is None:
        min_budget_increase = -current_tax_revenue * 0.015  # Allow 1.5% revenue increase

    print(f"  - max_budget_increase: {max_budget_increase:,.2f}")
    print(f"  - min_budget_increase: {min_budget_increase:,.2f}")

    # Add constraints (use solver_households, not original households)
    income_constraint = IncomeConstraint(income_constraint_value, list(solver_households.values()))
    budget_constraint = BudgetConstraint(
        "all_households",
        list(solver_households.values()),
        max_bln_mut_expenditure=max_budget_increase,
        min_bln_mut_expenditure=min_budget_increase,
    )
    marginal_pressure_constraint = MarginalPressureConstraint(max_marginal_pressure)

    tax_solver.add_constraints([
        income_constraint,
        budget_constraint,
        marginal_pressure_constraint,
    ])

    # These ensure certain rules can't be active at the same time
    mutually_exclusive_constraints = get_mutually_exclusive_constraints()
    tax_solver.add_constraints(mutually_exclusive_constraints)
    print(f"Added {len(mutually_exclusive_constraints)} mutually exclusive constraints")

    if objective_mode == "sequential":
        if budget_tolerance is None:
            budget_tolerance = min(current_tax_revenue * 0.01, max_budget_increase)
        objective = SequentialMixedObjective(
            budget_constraint,
            objectives={"budget": 2, "complexity": 1},
            tolerances={"budget": budget_tolerance, "complexity": 10},
        )
        print(f"  - budget_tolerance: {budget_tolerance:,.2f}")
    elif objective_mode == "budget_only":
        objective = BudgetObjective(budget_constraint)
    else:
        raise ValueError(f"Unknown objective_mode: {objective_mode}")
    tax_solver.add_objective(objective)

    # Solve; an infeasible model is reported, not raised
    print(f"\nSolving with max_marginal_pressure={max_marginal_pressure}...")
    try:
        tax_solver.solve()
    except ValueError:
        print("No feasible solution found for this configuration (guarantees jointly infeasible).")

    return tax_solver

In [ ]:
RUNS_DIR = "./systems/frontier_runs"
os.makedirs(RUNS_DIR, exist_ok=True)


def summarize_solution(tax_solver, current_tax_revenue: float, suffix: str = "") -> dict:
    """Extract the frontier outcomes and solver metadata from a (possibly
    unsolved) solver.

    Mirrors the summary bookkeeping of the batch cell in `fig09_10_dutch_case_study.ipynb`:
    revenue loss is the budget constraint's `spend` (positive = government
    collects less), the weighted rule count is sum(b * weight) over all rules.
    Keys are optionally suffixed so outcomes of several solves of the same
    sweep point can be merged into one record.
    """
    model = tax_solver.backend.model

    if tax_solver.solved:
        status = "optimal" if model.Status == 2 else "feasible"  # incumbent, gap may remain
    elif model.Status in (3, 4):
        status = "infeasible"
    else:
        status = "unresolved"

    out = {
        "status": status,
        "gurobi_status": model.Status,
        "runtime_s": round(model.Runtime, 2),
        "revenue_loss": np.nan,
        "revenue_loss_pct": np.nan,
        "weighted_rule_count": np.nan,
        "active_rules": np.nan,
        "mip_gap": np.nan,
    }

    if tax_solver.solved:
        r_and_r = tax_solver.rules_and_rates_table()
        out["active_rules"] = int(r_and_r["b"].sum())
        out["weighted_rule_count"] = int((r_and_r["b"] * r_and_r["weight"]).sum())

        for constraint in tax_solver.constraints:
            if hasattr(constraint, "spend"):
                out["revenue_loss"] = tax_solver.backend.get_value(constraint.spend)
                out["revenue_loss_pct"] = (
                    out["revenue_loss"] / abs(current_tax_revenue) * 100
                )
                break

        try:
            out["mip_gap"] = model.MIPGap
        except Exception:
            pass

    return {f"{k}{suffix}": v for k, v in out.items()}


def run_frontier_point(tag: str, **kwargs) -> dict:
    """Solve one sweep point in both objective modes and merge the outcomes.

    1. ``budget_only``: single-objective solve; its spend is the *true minimum*
       revenue loss under the guarantees (recorded with the ``_min`` suffix).
    2. ``sequential``: the paper's two-stage lexicographic procedure; gives the
       reform's weighted rule count and realized spend (unsuffixed keys).

    The rules-and-rates table of each solved model is written to
    ``./systems/frontier_runs/<tag>_<mode>.xlsx`` for later analysis. The
    sequential solve is skipped when the budget-only solve found no solution
    (feasibility is identical across modes because the constraints are the
    same); in that case the point's status is copied from the budget-only
    stage, so "unresolved" (time limit, no proof) stays distinguishable from
    proven "infeasible".
    """
    record = {}

    solver = run_nl_optimization(objective_mode="budget_only", **kwargs)
    record.update(summarize_solution(solver, current_tax_revenue, suffix="_min"))
    feasible = solver.solved
    if feasible:
        solver.rules_and_rates_table().to_excel(
            f"{RUNS_DIR}/{tag}_budget_only.xlsx", index=False
        )
    solver.close()

    if feasible:
        solver = run_nl_optimization(objective_mode="sequential", **kwargs)
        record.update(summarize_solution(solver, current_tax_revenue))
        if solver.solved:
            solver.rules_and_rates_table().to_excel(
                f"{RUNS_DIR}/{tag}_sequential.xlsx", index=False
            )
        solver.close()
    else:
        record["status"] = record["status_min"]

    return record

In [ ]:
# Load the NL data
data_path = "./data/NL_persons_headers_preprocessed_equal_weights.xlsx"
households = load_nl_data(data_path)
print(f"Total number of households loaded: {len(households)}")

# Weighted current tax revenue, used to size the budget band and to express
# revenue loss as a percentage of current revenue
current_tax_revenue = sum(
    hh.members[0]["weight"] * sum(
        p["income_before_tax"] - p["income_after_tax"] for p in hh.members
    )
    for hh in households.values()
)
print(f"Current tax revenue (weighted): {current_tax_revenue:,.2f}")

## Sweep 1: household income floor

The income floor is $1 - \texttt{income\_constraint\_value}$: sweeping the maximum allowed household income loss from 10% down to 0% covers floors $[0.90, 1.00]$, with denser steps near the tight end where the feasible set empties. The budget band stays at the Dutch-reform default of $\pm 1.5\%$ and the marginal-pressure cap at 65%.

In [ ]:
# Sweep 1: income floor over [0.90, 1.00], budget band fixed at +/-1.5%
income_loss_limits = [
    0.10, 0.09, 0.08, 0.07, 0.06, 0.05, 0.045, 0.04, 0.035, 0.03,
    0.025, 0.02, 0.015, 0.01, 0.005, 0.0,
]

os.makedirs("./systems", exist_ok=True)
floor_records = []
sweep_start = time.time()
for loss_limit in income_loss_limits:
    print(f"\n{'='*60}")
    print(f"Income floor {1 - loss_limit:.3f} (max income loss {loss_limit:.2%})")
    print(f"{'='*60}")

    record = {"income_floor": 1 - loss_limit, "income_loss_limit": loss_limit}
    record.update(run_frontier_point(
        tag=f"floor_{1 - loss_limit:.3f}",
        households=households,
        max_marginal_pressure=0.65,
        income_constraint_value=loss_limit,
        # budget band left at the +/-1.5% Dutch-reform default
    ))
    floor_records.append(record)

    # Save incrementally so an interrupted run keeps all finished points
    df_floor = pd.DataFrame(floor_records).sort_values("income_floor").reset_index(drop=True)
    df_floor.to_excel("./systems/case_frontier_income_floor.xlsx", index=False)

    print(f"\n-> floor {record['income_floor']:.3f}: {record['status']}, "
          f"min revenue loss {record['revenue_loss_pct_min']:.3f}%, "
          f"reform revenue loss {record.get('revenue_loss_pct', np.nan):.3f}%, "
          f"weighted rule count {record.get('weighted_rule_count', np.nan)} "
          f"[{time.time() - sweep_start:,.0f}s elapsed]")

df_floor

## Sweep 2: budget band

The symmetric budget band is swept from $\pm 3\%$ down to $\pm 0.5\%$ of current tax revenue, with the income floor fixed at the Dutch-reform value of 5% ($\texttt{income\_constraint\_value} = 0.05$) and the marginal-pressure cap at 65%. The budget tolerance of the second (complexity) objective stage is capped at the band width by `run_nl_optimization`, so the complexity stage cannot leave the swept band.

In [ ]:
# Sweep 2: symmetric budget band over [+/-0.25%, +/-3%], income floor fixed at 5%
budget_bands_pct = [3.0, 2.75, 2.5, 2.25, 2.0, 1.75, 1.5, 1.25, 1.0, 0.75, 0.5, 0.25]

band_records = []
sweep_start = time.time()
for band_pct in budget_bands_pct:
    band_abs = current_tax_revenue * band_pct / 100

    print(f"\n{'='*60}")
    print(f"Budget band +/-{band_pct}% (+/-{band_abs:,.0f})")
    print(f"{'='*60}")

    record = {"budget_band_pct": band_pct, "budget_band_abs": band_abs}
    record.update(run_frontier_point(
        tag=f"band_{band_pct:.2f}",
        households=households,
        max_marginal_pressure=0.65,
        income_constraint_value=0.05,
        max_budget_increase=band_abs,
        min_budget_increase=-band_abs,
    ))
    band_records.append(record)

    df_band = pd.DataFrame(band_records).sort_values("budget_band_pct").reset_index(drop=True)
    df_band.to_excel("./systems/case_frontier_budget_band.xlsx", index=False)

    print(f"\n-> band +/-{band_pct}%: {record['status']}, "
          f"min revenue loss {record['revenue_loss_pct_min']:.3f}%, "
          f"reform revenue loss {record.get('revenue_loss_pct', np.nan):.3f}%, "
          f"weighted rule count {record.get('weighted_rule_count', np.nan)} "
          f"[{time.time() - sweep_start:,.0f}s elapsed]")

df_band

## Sweeps 1b/2b: floor and band sweeps at additional marginal-pressure caps

The main sweeps hold the cap at the Dutch-reform value of 65%. Repeating them at a tight (55%) and a loose (75%) cap shows how the whole frontier shifts with the cap: where the feasibility cliff moves and how much complexity a given guarantee costs. Results are saved to cap-suffixed tables (`..._cap55.xlsx`, `..._cap75.xlsx`).

In [ ]:
# Sweeps 1b/2b: repeat floor and band sweeps at extra marginal-pressure caps
EXTRA_CAPS = [0.55, 0.75]

# Same grids as Sweeps 1 and 2 (redefined so this cell is self-contained)
floor_loss_limits = [
    0.10, 0.09, 0.08, 0.07, 0.06, 0.05, 0.045, 0.04, 0.035, 0.03,
    0.025, 0.02, 0.015, 0.01, 0.005, 0.0,
]
band_grid_pct = [3.0, 2.75, 2.5, 2.25, 2.0, 1.75, 1.5, 1.25, 1.0, 0.75, 0.5, 0.25]

for cap in EXTRA_CAPS:
    cap_id = f"{int(round(cap * 100))}"
    sweep_start = time.time()

    # --- Income floor sweep at this cap (band fixed at +/-1.5%) ---
    records = []
    for loss_limit in floor_loss_limits:
        print(f"\n{'='*60}")
        print(f"[cap {cap:.0%}] Income floor {1 - loss_limit:.3f} "
              f"(max income loss {loss_limit:.2%})")
        print(f"{'='*60}")

        record = {"marginal_pressure_cap": cap,
                  "income_floor": 1 - loss_limit,
                  "income_loss_limit": loss_limit}
        record.update(run_frontier_point(
            tag=f"floor_{1 - loss_limit:.3f}_cap{cap_id}",
            households=households,
            max_marginal_pressure=cap,
            income_constraint_value=loss_limit,
        ))
        records.append(record)

        df_out = pd.DataFrame(records).sort_values("income_floor").reset_index(drop=True)
        df_out.to_excel(f"./systems/case_frontier_income_floor_cap{cap_id}.xlsx", index=False)

        print(f"\n-> floor {record['income_floor']:.3f} @ cap {cap:.0%}: {record['status']} "
              f"[{time.time() - sweep_start:,.0f}s elapsed]")

    # --- Budget band sweep at this cap (floor fixed at 5%) ---
    records = []
    for band_pct in band_grid_pct:
        band_abs = current_tax_revenue * band_pct / 100
        print(f"\n{'='*60}")
        print(f"[cap {cap:.0%}] Budget band +/-{band_pct}%")
        print(f"{'='*60}")

        record = {"marginal_pressure_cap": cap,
                  "budget_band_pct": band_pct,
                  "budget_band_abs": band_abs}
        record.update(run_frontier_point(
            tag=f"band_{band_pct:.2f}_cap{cap_id}",
            households=households,
            max_marginal_pressure=cap,
            income_constraint_value=0.05,
            max_budget_increase=band_abs,
            min_budget_increase=-band_abs,
        ))
        records.append(record)

        df_out = pd.DataFrame(records).sort_values("budget_band_pct").reset_index(drop=True)
        df_out.to_excel(f"./systems/case_frontier_budget_band_cap{cap_id}.xlsx", index=False)

        print(f"\n-> band +/-{band_pct}% @ cap {cap:.0%}: {record['status']} "
              f"[{time.time() - sweep_start:,.0f}s elapsed]")

print("\nExtra-cap sweeps done")

## Resolve ambiguous points

A point is *ambiguous* when a solve hit the time limit without either retuning no solution found or the budget-only incumbent still had a large optimality gap. A wider band is used for those problems.

The cells below re-solve only those points with a much longer time limit and update the saved sweep tables in place.

In [ ]:
# Re-run ambiguous points with a longer per-solve time limit
RERUN_TIME_LIMIT = 2 * 60 * 60  # seconds per solve


def needs_rerun(row) -> bool:
    """Ambiguous = a stage hit the time limit without settling the question.

    Works on both the original overnight tables (where timeouts were labeled
    'infeasible') and re-run tables, by looking at the raw Gurobi status codes
    (9 = time limit) rather than the status labels.
    """
    # Budget-only stage timed out without any solution: feasibility unproven
    stage1_no_solution = row["gurobi_status_min"] == 9 and pd.isna(row["revenue_loss_pct_min"])
    # Budget-only incumbent still had a large optimality gap (>10%)
    stage1_poor_incumbent = pd.notna(row.get("mip_gap_min")) and row["mip_gap_min"] > 0.10
    # Sequential stage timed out without any solution
    seq_no_solution = row.get("gurobi_status") == 9 and pd.isna(row.get("weighted_rule_count"))
    seq_timeout_incumbent = (
        row.get("gurobi_status") == 9
        and pd.notna(row.get("weighted_rule_count"))
        and row.get("runtime_s", 0) < RERUN_TIME_LIMIT * 0.9
    )
    return bool(stage1_no_solution or stage1_poor_incumbent
                or seq_no_solution or seq_timeout_incumbent)


def rerun_sweep(path: str, kwargs_for_row) -> pd.DataFrame:
    """Re-solve every ambiguous row of a saved sweep table in place."""
    df = pd.read_excel(path)
    todo = [idx for idx, row in df.iterrows() if needs_rerun(row)]
    print(f"{path}: {len(todo)} ambiguous point(s) to re-run")

    for n, idx in enumerate(todo, 1):
        kwargs = kwargs_for_row(df.loc[idx])
        tag = kwargs.pop("tag")
        print(f"\n{'='*60}")
        print(f"[{n}/{len(todo)}] Re-running {tag} (time limit {RERUN_TIME_LIMIT / 3600:.0f}h)")
        print(f"{'='*60}")

        record = run_frontier_point(
            tag=tag, households=households, time_limit=RERUN_TIME_LIMIT, **kwargs
        )
        for k, v in record.items():
            df.loc[idx, k] = v
        df.to_excel(path, index=False)

        print(f"\n-> {tag}: {record['status']} "
              f"(min revenue loss {record['revenue_loss_pct_min']:.3f}%, "
              f"weighted rule count {record.get('weighted_rule_count', np.nan)})")

    return df


df_floor = rerun_sweep(
    "./systems/case_frontier_income_floor.xlsx",
    lambda r: dict(
        tag=f"floor_{r['income_floor']:.3f}",
        max_marginal_pressure=0.65,
        income_constraint_value=r["income_loss_limit"],
        # budget band left at the +/-1.5% Dutch-reform default
    ),
)

df_band = rerun_sweep(
    "./systems/case_frontier_budget_band.xlsx",
    lambda r: dict(
        tag=f"band_{r['budget_band_pct']:.2f}",
        max_marginal_pressure=0.65,
        income_constraint_value=0.05,
        max_budget_increase=r["budget_band_abs"],
        min_budget_increase=-r["budget_band_abs"],
    ),
)

# Also cover the extra-cap sweep tables (Sweeps 1b/2b), if they exist
for extra_cap in [0.55, 0.75]:
    cap_id = f"{int(round(extra_cap * 100))}"

    floor_path = f"./systems/case_frontier_income_floor_cap{cap_id}.xlsx"
    if os.path.exists(floor_path):
        rerun_sweep(floor_path, lambda r, cap=extra_cap, cid=cap_id: dict(
            tag=f"floor_{r['income_floor']:.3f}_cap{cid}",
            max_marginal_pressure=cap,
            income_constraint_value=r["income_loss_limit"],
        ))

    band_path = f"./systems/case_frontier_budget_band_cap{cap_id}.xlsx"
    if os.path.exists(band_path):
        rerun_sweep(band_path, lambda r, cap=extra_cap, cid=cap_id: dict(
            tag=f"band_{r['budget_band_pct']:.2f}_cap{cid}",
            max_marginal_pressure=cap,
            income_constraint_value=0.05,
            max_budget_increase=r["budget_band_abs"],
            min_budget_increase=-r["budget_band_abs"],
        ))

## Figure: frontiers across marginal-pressure caps

Complexity frontiers (weighted rule count of the lexicographic reform) against the income floor and the budget band, one line per cap. Shows how the feasibility cliff and the complexity price of a guarantee move with the cap. Only caps whose sweep tables exist on disk are drawn, so this cell works before, during, and after the extra-cap overnight run.

In [ ]:
# Multi-cap comparison: complexity frontier per marginal-pressure cap
from matplotlib.lines import Line2D

# Okabe-Ito colorblind-safe palette, one color per cap. The 65% cap is the
# paper's main-result setting and is drawn heavier.
CAP_STYLES = {
    0.55: {"color": "#D55E00", "marker": "^", "lw": 2},    # vermillion
    0.65: {"color": "#0072B2", "marker": "o", "lw": 3.2},  # blue (main result)
    0.75: {"color": "#009E73", "marker": "s", "lw": 2},    # green
}

# Main-result settings of the Dutch reform (Section: the Dutch case study)
SQ_CAP, SQ_FLOOR, SQ_BAND = 0.65, 0.95, 1.5


def cap_table(kind: str, cap: float) -> str:
    """Path of the sweep table for a given cap (0.65 = the main sweep files)."""
    if np.isclose(cap, 0.65):
        return f"./systems/case_frontier_{kind}.xlsx"
    return f"./systems/case_frontier_{kind}_cap{int(round(cap * 100))}.xlsx"


def draw_infeasible_whisker(ax, x0, x1, y_level, color, tick_height):
    """Boxplot-style whisker at a fixed y spanning a proven-infeasible x-range,
    anchored at the last achieved weighted rule count (reads as "the frontier
    would continue at this level, but is infeasible from here on").
    """
    ax.plot([x0, x1], [y_level, y_level], color=color, lw=2,
            linestyle=(0, (4, 2)), solid_capstyle="butt", zorder=3)
    for x in (x0, x1):
        ax.plot([x, x], [y_level - tick_height, y_level + tick_height],
                color=color, lw=2, zorder=3)


def draw_frontier_lines(ax, kind: str, xcol: str, band_edge_for_row) -> dict:
    """Draw the feasible frontier for every cap; return each cap's whisker
    span (x0, x1, y_level) anchored at the last recorded weighted rule count.
    Markers are filled where the budget-only optimum attains the band's
    revenue edge (``band_edge_for_row``) and open where the marginal-pressure
    cap binds instead of the band.
    """
    whiskers = {}
    for cap, style in CAP_STYLES.items():
        path = cap_table(kind, cap)
        if not os.path.exists(path):
            continue
        df = pd.read_excel(path)
        feas = df[df["status"].isin(["optimal", "feasible"])].sort_values(xcol)
        infeas = df[df["status"] == "infeasible"].sort_values(xcol)

        marker = style["marker"]
        line_style = {k: v for k, v in style.items() if k != "marker"}
        ax.plot(feas[xcol], feas["weighted_rule_count"], **line_style)

        attains = np.isclose(
            feas["revenue_loss_pct_min"].astype(float),
            feas.apply(band_edge_for_row, axis=1).astype(float),
            atol=0.01,
        )
        ax.plot(feas.loc[attains, xcol], feas.loc[attains, "weighted_rule_count"],
                marker=marker, linestyle="none", color=style["color"])
        ax.plot(feas.loc[~attains, xcol], feas.loc[~attains, "weighted_rule_count"],
                marker=marker, linestyle="none", markerfacecolor="white",
                markeredgecolor=style["color"], markeredgewidth=1.8, zorder=4)

        if not infeas.empty and not feas.empty:
            last_feasible = feas.iloc[-1]  # closest to the cliff
            whiskers[cap] = (
                float(last_feasible[xcol]),
                float(infeas[xcol].max()),
                float(last_feasible["weighted_rule_count"]),
            )
    return whiskers


fig, (ax_floor_c, ax_band_c) = plt.subplots(1, 2, figsize=(14, 6))

# Band edge: the floor sweep holds the band at +/-1.5%; the band sweep's edge
# is the swept band itself
whiskers_floor = draw_frontier_lines(
    ax_floor_c, "income_floor", "income_floor", lambda r: -1.5
)
draw_frontier_lines(
    ax_band_c, "budget_band", "budget_band_pct", lambda r: -r["budget_band_pct"]
)

# Mark the main-result operating point (star on the 65% frontier)
for ax, kind, xcol, x_sq in [
    (ax_floor_c, "income_floor", "income_floor", SQ_FLOOR),
    (ax_band_c, "budget_band", "budget_band_pct", SQ_BAND),
]:
    df65 = pd.read_excel(cap_table(kind, SQ_CAP))
    row = df65[np.isclose(df65[xcol].astype(float), x_sq)]
    if not row.empty:
        ax.plot(
            row[xcol], row["weighted_rule_count"],
            marker="*", markersize=22, color="black", linestyle="none", zorder=6,
        )
    ax.axvline(x_sq, color="black", linestyle=":", lw=1.2)

ax_floor_c.set_xlim(0.895, 1.005)  # include the fully infeasible floors up to 1.00

# Whiskers at the last-achieved rule count, extended across the infeasible range
tick_height = 0.018 * np.diff(ax_floor_c.get_ylim())[0]
for cap, (x0, x1, y_level) in whiskers_floor.items():
    draw_infeasible_whisker(ax_floor_c, x0, x1, y_level, CAP_STYLES[cap]["color"], tick_height)

ax_floor_c.set_xlabel("Household income floor (share of current net income)", fontsize=14)
ax_floor_c.set_ylabel("Weighted rule count", fontsize=13)
ax_floor_c.set_title("Income floor sweep (budget band fixed at \u00b11.5%)", fontsize=14)
ax_floor_c.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)


ax_band_c.set_xlabel("Budget band (\u00b1 % of current revenue)", fontsize=14)
ax_band_c.set_ylabel("Weighted rule count", fontsize=13)
ax_band_c.set_title("Budget band sweep (income floor fixed at 5%)", fontsize=14)
ax_band_c.grid(True, linestyle="--", linewidth=0.7, alpha=0.6)

# Shared legend: cap lines (proxy handles, since data markers are drawn in
# separate filled/open layers) + star, whiskers, and the open-marker convention
handles = [
    Line2D([], [], marker=s["marker"], color=s["color"], lw=s["lw"])
    for s in CAP_STYLES.values()
]
labels = [f"{cap:.0%} cap" for cap in CAP_STYLES]
handles += [
    Line2D([], [], marker="*", markersize=16, color="black", linestyle="none"),
    Line2D([], [], color="grey", lw=2, linestyle=(0, (4, 2))),
    Line2D([], [], marker="o", linestyle="none", markerfacecolor="white",
           markeredgecolor="grey", markeredgewidth=1.8),
]
labels += [
    "Main-result settings",
    "Proven infeasible",
    "Open: budget not at cap",
]
fig.legend(handles, labels, loc="lower center", ncol=3, fontsize=11.5,
           frameon=False, bbox_to_anchor=(0.5, -0.08))

plt.tight_layout()
fig.savefig("./output/figs/case_frontier_caps.png", dpi=300, bbox_inches="tight")
print("Figure saved to ./output/figs/case_frontier_caps.png")
plt.show()

## Optional: additional cap sweep (not required for either figure)

Marginal-pressure cap swept over [0.55, 0.80] at the status-quo floor/band. 
Not needed for paper figure.

In [ ]:
# Sweep 3: marginal-pressure cap over [0.55, 0.80], floor and band at status quo
pressure_caps = [0.80, 0.75, 0.70, 0.65, 0.625, 0.60, 0.575, 0.55]

cap_records = []
sweep_start = time.time()
for cap in pressure_caps:
    print(f"\n{'='*60}")
    print(f"Marginal-pressure cap {cap:.1%}")
    print(f"{'='*60}")

    record = {"marginal_pressure_cap": cap}
    record.update(run_frontier_point(
        tag=f"cap_{cap:.3f}",
        households=households,
        max_marginal_pressure=cap,
        income_constraint_value=0.05,
        # budget band left at the +/-1.5% Dutch-reform default
    ))
    cap_records.append(record)

    df_cap = pd.DataFrame(cap_records).sort_values("marginal_pressure_cap").reset_index(drop=True)
    df_cap.to_excel("./systems/case_frontier_pressure_cap.xlsx", index=False)

    print(f"\n-> cap {cap:.1%}: {record['status']}, "
          f"min revenue loss {record['revenue_loss_pct_min']:.3f}%, "
          f"reform revenue loss {record.get('revenue_loss_pct', np.nan):.3f}%, "
          f"weighted rule count {record.get('weighted_rule_count', np.nan)} "
          f"[{time.time() - sweep_start:,.0f}s elapsed]")

df_cap

In [ ]:
# Resolve ambiguous points in the optional cap sweep, same as the main sweeps above
df_cap = rerun_sweep(
    "./systems/case_frontier_pressure_cap.xlsx",
    lambda r: dict(
        tag=f"cap_{r['marginal_pressure_cap']:.3f}",
        max_marginal_pressure=r["marginal_pressure_cap"],
        income_constraint_value=0.05,
    ),
)
